# Image Enhancer CNN

Trains a small CNN on `data/processed/images` and `data/processed/labels.csv`. Input is RGB `[1, 3, 224, 224]`; output is `[brightness, contrast, saturation]` correction parameters.

In [1]:
import csv
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

try:
    RESAMPLE_BILINEAR = Image.Resampling.BILINEAR
except AttributeError:
    RESAMPLE_BILINEAR = Image.BILINEAR

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 12
LR = 1e-3
VAL_FRACTION = 0.1

ROOT = Path.cwd()
if ROOT.name != "ml" and (ROOT / "ml").exists():
    ROOT = ROOT / "ml"

DATA_DIR = ROOT / "data" / "processed"
IMAGE_DIR = DATA_DIR / "images"
LABELS_CSV = DATA_DIR / "labels.csv"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Device: {DEVICE}")
print(f"Dataset: {DATA_DIR}")

Device: cpu
Dataset: c:\Users\Марсель\Documents\practice\ml\data\processed


In [2]:
class EnhancementCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 96, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        self.head = nn.Sequential(
            nn.Linear(96, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 3),
        )

    def forward(self, image):
        return self.head(self.features(image))


model = EnhancementCNN().to(DEVICE)
model(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)).shape

torch.Size([1, 3])

In [3]:
class EnhancementDataset(Dataset):
    def __init__(self, labels_csv=LABELS_CSV, image_dir=IMAGE_DIR):
        self.image_dir = Path(image_dir)
        with Path(labels_csv).open("r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            expected = ["image_id", "brightness", "contrast", "saturation"]
            if reader.fieldnames != expected:
                raise ValueError(f"Expected labels header {expected}, got {reader.fieldnames}")
            self.rows = list(reader)

        if not self.rows:
            raise ValueError(f"No labels found in {labels_csv}")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        image_path = self.image_dir / f"{row['image_id']}.jpg"
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            if image.size != (IMAGE_SIZE, IMAGE_SIZE):
                image = image.resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE_BILINEAR)
            array = np.asarray(image, dtype=np.float32) / 255.0

        tensor = torch.from_numpy(array).permute(2, 0, 1)
        target = torch.tensor([
            float(row["brightness"]),
            float(row["contrast"]),
            float(row["saturation"]),
        ], dtype=torch.float32)
        return tensor, target


dataset = EnhancementDataset()
indices = list(range(len(dataset)))
random.Random(SEED).shuffle(indices)
val_size = max(1, int(len(indices) * VAL_FRACTION))
val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(DEVICE == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(DEVICE == "cuda"))

print(f"Samples: {len(dataset)} train={len(train_dataset)} val={len(val_dataset)}")
next(iter(train_loader))[0].shape, next(iter(train_loader))[1].shape

Samples: 5000 train=4500 val=500


(torch.Size([64, 3, 224, 224]), torch.Size([64, 3]))

In [4]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_fn = nn.SmoothL1Loss()
    total_loss = 0.0
    total_mae = torch.zeros(3, device=DEVICE)
    count = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for image, target in loader:
            image = image.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            pred = model(image)
            loss = loss_fn(pred, target)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            batch_size = image.size(0)
            total_loss += loss.item() * batch_size
            total_mae += torch.abs(pred.detach() - target).sum(dim=0)
            count += batch_size

    return total_loss / count, (total_mae / count).detach().cpu().tolist()


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_state = None
    best_val = float("inf")

    for epoch in tqdm(range(epochs), desc="Training"):
        train_loss, train_mae = run_epoch(model, train_loader, optimizer)
        val_loss, val_mae = run_epoch(model, val_loader)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        tqdm.write(
            f"Epoch {epoch + 1:02d} "
            f"train_loss={train_loss:.5f} val_loss={val_loss:.5f} "
            f"val_mae=[b={val_mae[0]:.4f}, c={val_mae[1]:.4f}, s={val_mae[2]:.4f}]"
        )

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


model = train_model(model, train_loader, val_loader)

Training:   0%|          | 0/12 [00:00<?, ?it/s]

Epoch 01 train_loss=0.17957 val_loss=0.08938 val_mae=[b=0.1075, c=0.4357, s=0.4262]
Epoch 02 train_loss=0.07377 val_loss=0.06107 val_mae=[b=0.0928, c=0.2936, s=0.4012]
Epoch 03 train_loss=0.06201 val_loss=0.05778 val_mae=[b=0.0907, c=0.2670, s=0.3979]
Epoch 04 train_loss=0.05914 val_loss=0.05478 val_mae=[b=0.0852, c=0.2505, s=0.3920]
Epoch 05 train_loss=0.05671 val_loss=0.05412 val_mae=[b=0.0826, c=0.2554, s=0.3853]
Epoch 06 train_loss=0.05455 val_loss=0.05258 val_mae=[b=0.0832, c=0.2334, s=0.3899]
Epoch 07 train_loss=0.05320 val_loss=0.05120 val_mae=[b=0.0783, c=0.2423, s=0.3799]
Epoch 08 train_loss=0.05066 val_loss=0.05033 val_mae=[b=0.0855, c=0.2220, s=0.3826]
Epoch 09 train_loss=0.04915 val_loss=0.05015 val_mae=[b=0.0774, c=0.2338, s=0.3780]
Epoch 10 train_loss=0.04926 val_loss=0.05059 val_mae=[b=0.0749, c=0.2444, s=0.3744]
Epoch 11 train_loss=0.04733 val_loss=0.04497 val_mae=[b=0.0771, c=0.2070, s=0.3648]
Epoch 12 train_loss=0.04780 val_loss=0.04582 val_mae=[b=0.0732, c=0.2031, s=

In [5]:
def export_onnx(model, path):
    model.eval().cpu()
    dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
    torch.onnx.export(
        model,
        dummy,
        path,
        input_names=["image"],
        output_names=["params"],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )
    return path


checkpoint_path = MODEL_DIR / "enhancer.onnx"
export_onnx(model, checkpoint_path)
checkpoint_path

C:\Users\Марсель\AppData\Local\Temp\ipykernel_8184\1414930405.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


WindowsPath('c:/Users/Марсель/Documents/practice/ml/models/enhancer.onnx')